# 07 â€” Dashboard: herramienta de exploraciÃ³n de perfiles

**Objetivo:** construir la herramienta de apoyo a la decisiÃ³n que consume los perfiles generados
en `06_clustering` (K-Means, K=5) y, opcionalmente, la representaciÃ³n semÃ¡ntica de `07_embeddings`,
para los casos de uso descritos en `context/00-contexto-permanente.md`: asignaciÃ³n de tareas,
conformaciÃ³n de comisiones/equipos, planificaciÃ³n acadÃ©mica y administrativa.

**DecisiÃ³n de herramienta (ver DEC-003 en `context/DECISION_LOG.md`):** el prototipo se construye
como una aplicaciÃ³n **Streamlit** (`app.py`, en esta misma carpeta), no como un notebook con
widgets. Motivo: el contexto permanente del proyecto pide explÃ­citamente un "prototipo/dashboard"
pensado desde el inicio para "eventual despliegue, mantenimiento, actualizaciÃ³n de datos" y para
apoyar a unidades institucionales (RRHH, decanatos) que no son usuarios tÃ©cnicos â€” un notebook no
es la interfaz adecuada para ese pÃºblico ni para ese ciclo de vida. Este notebook
(`08_dashboard.ipynb`) se usa entonces para:

1. Preparar y validar los datasets consolidados que consume la aplicaciÃ³n (`data/dashboard/`).
2. Verificar rÃ¡pidamente que los clusters no son un proxy trivial de variables demogrÃ¡ficas
   (sexo, edad) â€” relevante para las restricciones conceptuales del proyecto.
3. Documentar y probar la bÃºsqueda semÃ¡ntica opcional sobre `embeddings_personas.csv`.
4. Dejar instrucciones para ejecutar la aplicaciÃ³n.

**Entradas:**

- `data/clustering/clusters_personas.csv`, `cluster_sizes.csv`, `cluster_characterization.csv`
  (de `06_clustering`).
- `data/features/dataset_personas_features.csv`, `feature_dictionary.csv`, `personas.csv`.
- `data/embeddings/embeddings_personas.csv`, `corpus_texto_detalle.csv`,
  `cobertura_texto_personas.csv` (de `07_embeddings`).

**Salidas (en `data/dashboard/`):** `personas_dashboard.csv`, `cluster_perfiles_resumen.csv`,
`cluster_top_features.csv`.

**Nota de entorno:** este notebook usa un entorno virtual aislado (`.venv/`, kernel
*"Proyecto Tesis - Dashboard (.venv)"*) en vez del intÃ©rprete global usado por los notebooks 01-06
(kernel *"proyecto-tesis-py311"*), para poder instalar `streamlit`/`plotly` sin generar conflictos
de dependencias con otras herramientas ya instaladas globalmente (ver DEC-003).

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd().parents[1] if (Path.cwd().name == "08_dashboard") else Path.cwd()
sys.path.insert(0, str(Path.cwd()))
import lib

DATA_FEATURES = ROOT / "data" / "features"
DATA_CLUSTERING = ROOT / "data" / "clustering"
DATA_EMBEDDINGS = ROOT / "data" / "embeddings"
DATA_DASHBOARD = ROOT / "data" / "dashboard"
DATA_DASHBOARD.mkdir(parents=True, exist_ok=True)

print("Salidas del dashboard en:", DATA_DASHBOARD)

## 1. ConsolidaciÃ³n de datos por persona

Se une `clusters_personas.csv` (asignaciÃ³n de cluster) con las 85 features originales de
`dataset_personas_features.csv` y la cobertura de texto de `07_embeddings`, y se agrega el nombre
de perfil (secciÃ³n 8 de `06_clustering`, sintetizado en `lib.PERFIL_NOMBRES`).

In [ ]:
clusters = pd.read_csv(DATA_CLUSTERING / "clusters_personas.csv")
features = pd.read_csv(DATA_FEATURES / "dataset_personas_features.csv")
cobertura = pd.read_csv(DATA_EMBEDDINGS / "cobertura_texto_personas.csv")
roles_simultaneos = pd.read_csv(
    ROOT / "data" / "trayectorias" / "features_trayectoria_persona.csv",
    usecols=["IDPERSONA", "TUVO_ROL_ADICIONAL_SIMULTANEO", "CATEGORIA_ROL_ADICIONAL_MAS_RECIENTE",
             "TUVO_FUNCION_ADICIONAL", "CATEGORIA_FUNCION_ADICIONAL_MAS_RECIENTE",
             "CARGO_ACTUAL_ESTRUCTURAL", "UNIDAD_ACTUAL_ESTRUCTURAL", "VIGENTE_TRAMO_ESTRUCTURAL",
             "CARGO_PUNTUAL_VIGENTE", "UNIDAD_PUNTUAL_VIGENTE", "CATEGORIA_PUNTUAL_VIGENTE",
             "ES_MIXTO", "CARGOS_ACTUALES_MIXTO", "CATEGORIAS_ACTUALES_MIXTO",
             "ANIOS_EN_UNIDAD_ACTUAL", "DURACION_MEDIANA_TRAMO_ANIOS", "TURBULENCIA_TRAMOS",
             "ENTROPIA_CATEGORIA_CARGO", "PROPORCION_ANIOS_ADMINISTRATIVO"],
)

assert set(clusters["IDPERSONA"]) == set(features["IDPERSONA"]), "IDPERSONA no coincide entre clustering y features"

# DEC-010: features.TIPOEMPLEADO_ACTUAL_DESC (ultimo contrato por fecha) puede no coincidir
# de rama con clusters.GRUPO_PRINCIPAL (tramo de rol vigente, la misma fuente que CLUSTER) -
# se descarta la version de features y se usa GRUPO_PRINCIPAL como TIPOEMPLEADO_ACTUAL_DESC
# para que la composicion por tipo de empleado de cada perfil sea 100/0 por construccion.
features = features.drop(columns=["TIPOEMPLEADO_ACTUAL_DESC"])
# DEC-016: mismo problema de raiz para CARGO_ACTUAL/UNIDAD_ACTUAL_NOMBRE (calculados por
# separado sobre el ultimo contrato por fecha, sin excluir categorias puntuales/de ruido -
# pueden mostrar un cargo de un tramo distinto al que determina CATEGORIA_CARGO_ACTUAL/
# CLUSTER, p.ej. una subrogacion breve y reciente en vez del cargo estructural continuo).
# Se descartan y se reemplazan por CARGO_ACTUAL_ESTRUCTURAL/UNIDAD_ACTUAL_ESTRUCTURAL, que
# vienen del mismo tramo que CATEGORIA_CARGO_ACTUAL (ver construir_tramos_rol en
# _preprocesamiento_comun.py y DEC-016 en context/DECISION_LOG.md).
features = features.drop(columns=["CARGO_ACTUAL", "UNIDAD_ACTUAL_NOMBRE"])

personas_dashboard = clusters.merge(features, on="IDPERSONA", how="left")
personas_dashboard = personas_dashboard.merge(cobertura, on="IDPERSONA", how="left")
# DEC-011: rol adicional ejercido en paralelo (p.ej. autoridad concurrente con un cargo
# estructural continuo) - se expone como atributo/afinidad, no cambia CLUSTER/SUBGRUPO.
# DEC-012: idem para funciones adicionales/subrogaciones explicitas (registro_autoridades).
personas_dashboard = personas_dashboard.merge(roles_simultaneos, on="IDPERSONA", how="left")
personas_dashboard["TUVO_ROL_ADICIONAL_SIMULTANEO"] = personas_dashboard["TUVO_ROL_ADICIONAL_SIMULTANEO"].fillna(False)
personas_dashboard["TUVO_FUNCION_ADICIONAL"] = personas_dashboard["TUVO_FUNCION_ADICIONAL"].fillna(False)
# Correccion 2026-09-15 (caso IDPERSONA 3519, ver DECISION_LOG.md): ES_MIXTO=True cuando la
# persona tiene 2+ cargos estructurales vigentes en paralelo (no subrogaciones) - atributo
# de presentacion, no cambia CLUSTER/CATEGORIA_CARGO_ACTUAL. CARGOS_ACTUALES_MIXTO/
# CATEGORIAS_ACTUALES_MIXTO no necesitan fillna (NaN = no mixto, mismo tratamiento que
# CATEGORIA_ROL_ADICIONAL_MAS_RECIENTE).
personas_dashboard["ES_MIXTO"] = personas_dashboard["ES_MIXTO"].fillna(False)
personas_dashboard = personas_dashboard.rename(columns={
    "GRUPO_PRINCIPAL": "TIPOEMPLEADO_ACTUAL_DESC",
    "CARGO_ACTUAL_ESTRUCTURAL": "CARGO_ACTUAL",
    "UNIDAD_ACTUAL_ESTRUCTURAL": "UNIDAD_ACTUAL_NOMBRE",
})
# DEC-017 (ver context/DECISION_LOG.md): si la persona no tiene tramo estructural vigente
# hoy (VIGENTE_TRAMO_ESTRUCTURAL=False) pero SI tiene un contrato puntual vigente (p.ej.
# servicios profesionales por proyecto, excluido del tramo estructural por diseño desde
# DEC-004), se usa ese contrato puntual como CARGO_ACTUAL/UNIDAD_ACTUAL_NOMBRE - de lo
# contrario el dashboard muestra un cargo estructural ya finalizado como si fuera actual,
# contradiciendo VIGENTE_ACTUALMENTE=True. No cambia CLUSTER/CATEGORIA_CARGO_ACTUAL, que
# siguen basados unicamente en el tramo estructural (correcto para el perfilamiento).
_usar_puntual = (personas_dashboard["VIGENTE_TRAMO_ESTRUCTURAL"] != True) & personas_dashboard["CARGO_PUNTUAL_VIGENTE"].notna()  # noqa: E712
personas_dashboard["CARGO_ES_CONTRATO_PUNTUAL_VIGENTE"] = _usar_puntual
personas_dashboard.loc[_usar_puntual, "CARGO_ACTUAL"] = personas_dashboard.loc[_usar_puntual, "CARGO_PUNTUAL_VIGENTE"]
personas_dashboard.loc[_usar_puntual, "UNIDAD_ACTUAL_NOMBRE"] = personas_dashboard.loc[_usar_puntual, "UNIDAD_PUNTUAL_VIGENTE"]
personas_dashboard = personas_dashboard.drop(columns=["CARGO_PUNTUAL_VIGENTE", "UNIDAD_PUNTUAL_VIGENTE", "CATEGORIA_PUNTUAL_VIGENTE"])
personas_dashboard["PERFIL_NOMBRE"] = personas_dashboard["CLUSTER"].map(lib.PERFIL_NOMBRES)

cols = ["IDPERSONA", "CLUSTER", "PERFIL_NOMBRE"] + [
    c for c in personas_dashboard.columns if c not in ("IDPERSONA", "CLUSTER", "PERFIL_NOMBRE")
]
personas_dashboard = personas_dashboard[cols]

personas_dashboard.to_csv(DATA_DASHBOARD / "personas_dashboard.csv", index=False)
print(personas_dashboard.shape)
personas_dashboard.head(3)

**Nota sobre valores faltantes:** columnas como `PROMEDIO_ESTUDIANTES_POR_CURSO`,
`PROMEDIO_HETEROEVALUACION` o `DEDICACION_DOCENTE_ACTUAL` tienen NaN para personas sin actividad
docente (no aplica), no un dato perdido por error de captura â€” igual que en `dataset_personas_features.csv`
(ver `data/features/feature_dictionary.csv`). La aplicaciÃ³n debe mostrar estos casos como
"no aplica", no como "sin dato".

## 2. Resumen por cluster (para tarjetas de perfil en la app)

In [ ]:
sizes = pd.read_csv(DATA_CLUSTERING / "cluster_sizes.csv")
resumen = sizes.copy()
resumen["PERFIL_NOMBRE"] = resumen["CLUSTER"].map(lib.PERFIL_NOMBRES)
resumen["DESCRIPCION"] = resumen["CLUSTER"].map(lib.PERFIL_DESCRIPCIONES)
resumen.to_csv(DATA_DASHBOARD / "cluster_perfiles_resumen.csv", index=False)
resumen

In [ ]:
charac = pd.read_csv(DATA_CLUSTERING / "cluster_characterization.csv")
top_features = (
    charac.sort_values(["CLUSTER", "IMPORTANCE"], ascending=[True, False])
    .groupby("CLUSTER")
    .head(8)
    .reset_index(drop=True)
)
top_features.to_csv(DATA_DASHBOARD / "cluster_top_features.csv", index=False)
top_features.head(10)

## 3. VerificaciÃ³n visual: tamaÃ±o y composiciÃ³n de los perfiles

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
order = resumen.sort_values("CLUSTER")
ax.bar(order["PERFIL_NOMBRE"], order["N_PERSONAS"], color=sns.color_palette("Set2", len(order)))
ax.set_ylabel("N personas")
ax.set_title("TamaÃ±o de cada perfil")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
tab_tipo = pd.crosstab(personas_dashboard["PERFIL_NOMBRE"], personas_dashboard["TIPOEMPLEADO_ACTUAL_DESC"], normalize="index") * 100
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(tab_tipo.round(1), annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% dentro del perfil"}, ax=ax)
ax.set_title("ComposiciÃ³n por tipo de empleado (%)")
plt.tight_layout()
plt.show()
tab_tipo.round(1)

Confirma lo documentado en `06_clustering` (secciÃ³n 8): el perfil **Administrativo** concentra casi
todo el personal administrativo, mientras que los otros cuatro perfiles son mayoritariamente
docentes â€” coherente con los nombres propuestos, sin ser una regla perfecta (algunas personas con
`ES_DOCENTE_ADMIN_MIXTO=True` aparecen en perfiles docentes).

## 4. Chequeo de equidad: Â¿los perfiles son un proxy de sexo o edad?

El proyecto exige explÃ­citamente no presentar los perfiles como verdades absolutas y evitar que
sirvan de base indirecta para decisiones sensibles (ver `context/00-contexto-permanente.md`,
restricciones conceptuales). `SEXO` y `EDAD` **no se usaron como variables de clustering** (no
estÃ¡n en `X_modelado.csv`); aquÃ­ se verifica, solo como chequeo de sanidad, que la distribuciÃ³n de
estas variables no varÃ­e de forma extrema entre perfiles â€” si lo hiciera, serÃ­a una seÃ±al de alerta
sobre variables proxy dentro de las features usadas (antigÃ¼edad, tipo de cargo, etc.) que ameritarÃ­a
revisiÃ³n adicional antes de usar los perfiles operativamente.

In [ ]:
demograficos = pd.read_csv(DATA_FEATURES / "personas.csv")[["IDPERSONA", "SEXO", "EDAD"]]
check = personas_dashboard[["IDPERSONA", "CLUSTER", "PERFIL_NOMBRE"]].merge(demograficos, on="IDPERSONA", how="left")

tab_sexo = pd.crosstab(check["PERFIL_NOMBRE"], check["SEXO"], normalize="index") * 100
display(tab_sexo.round(1))

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=check, x="PERFIL_NOMBRE", y="EDAD", ax=ax)
ax.set_title("DistribuciÃ³n de edad por perfil")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

**Lectura:** ni `SEXO` ni `EDAD` se usan como filtro ni como variable en la aplicaciÃ³n â€” se
muestran aquÃ­ Ãºnicamente como chequeo de equidad, no como parte de la ficha de persona ni de los
criterios de bÃºsqueda de la secciÃ³n "Formar equipos" de la app, precisamente para no introducir
esas variables como criterio de selecciÃ³n de personal.

## 5. Búsqueda semántica (opcional) — prueba de concepto

Se prueba la búsqueda semántica de texto libre sobre `embeddings_personas.csv` (768
dimensiones, `intfloat/multilingual-e5-base`, DEC-014 — reemplaza el modelo/enfoque de
DEC-002): cada persona tiene un único embedding calculado sobre su documento semántico
completo (`documento_semantico_persona.csv`), no un promedio de fragmentos. La consulta se
codifica con el mismo modelo, con el prefijo `"query: "` que requiere E5 (los documentos se
codificaron con `"passage: "`, ver `07_embeddings.ipynb`), y se calcula similitud coseno por
fuerza bruta con `numpy`. La aplicación (`app.py`) reutiliza esta misma lógica
(`lib.load_embeddings`).

In [ ]:
from sentence_transformers import SentenceTransformer

ids, X = lib.load_embeddings()
modelo_texto = SentenceTransformer("intfloat/multilingual-e5-base")

def buscar_personas(consulta, top_n=5):
    q = modelo_texto.encode([f"query: {consulta}"], normalize_embeddings=True)[0]
    sims = X @ q
    top_idx = np.argsort(-sims)[:top_n]
    return pd.DataFrame({"IDPERSONA": ids[top_idx], "SIMILITUD": sims[top_idx]})

ejemplo = buscar_personas("experiencia en inteligencia artificial aplicada a imágenes médicas")
ejemplo = ejemplo.merge(personas_dashboard[["IDPERSONA", "CLUSTER", "PERFIL_NOMBRE"]], on="IDPERSONA", how="left")
ejemplo

**Limitaciones de la bÃºsqueda semÃ¡ntica** (mostrar siempre en la app junto a los resultados):

- Solo 2196 de 2213 personas (99.2%) tienen algÃºn texto fuente; el resto no puede aparecer en estos
  resultados aunque sea relevante (columna `N_REGISTROS_TEXTO=0` en `cobertura_texto_personas.csv`).
- La similitud semÃ¡ntica **no es una medida de idoneidad**: una alta similitud indica que el texto
  histÃ³rico de esa persona (publicaciones, capacitaciones, ponencias, etc.) se relaciona
  temÃ¡ticamente con la consulta, no que sea la mejor opciÃ³n para una tarea â€” requiere criterio
  humano, igual que el resto de la herramienta.

## 6. Resumen y cÃ³mo ejecutar la aplicaciÃ³n

**Archivos generados en `data/dashboard/`:**

| Archivo | Contenido |
|---|---|
| `personas_dashboard.csv` | Una fila por persona: `IDPERSONA`, `CLUSTER`, `PERFIL_NOMBRE` + 85 features originales + cobertura de texto |
| `cluster_perfiles_resumen.csv` | TamaÃ±o, % de poblaciÃ³n, nombre y descripciÃ³n por cluster |
| `cluster_top_features.csv` | Top 8 variables mÃ¡s distintivas por cluster (de `cluster_characterization.csv`) |

**CÃ³mo ejecutar la aplicaciÃ³n (`app.py`, en esta misma carpeta):**

```
d:\Proyecto_Tesis\.venv\Scripts\streamlit run notebooks\08_dashboard\app.py
```

La primera vez que se use la pestaÃ±a de bÃºsqueda semÃ¡ntica, la app carga el modelo de embeddings
en memoria (unos segundos); ya estÃ¡ cacheado localmente por `07_embeddings`, por lo que no requiere
conexiÃ³n a internet.

**Limitaciones y advertencias:**

- Los nombres/descripciones de perfil son una sÃ­ntesis interpretativa (secciÃ³n 8 de
  `06_clustering`), no una categorÃ­a institucional oficial ni una verdad absoluta sobre las
  personas â€” la app debe mostrar este recordatorio de forma permanente (`lib.DISCLAIMER`).
- La app y sus datasets solo usan `IDPERSONA` (identificador ya anonimizado en el proyecto); no
  incorpora nombres reales. Un eventual despliegue institucional requerirÃ­a integrarse con los
  sistemas de RRHH bajo el control de acceso correspondiente â€” algo fuera del alcance de este
  prototipo.
- `SEXO` y `EDAD` se revisaron solo como chequeo de equidad (secciÃ³n 4) y **no** se usan como
  filtro ni criterio dentro de la aplicaciÃ³n.
- La bÃºsqueda semÃ¡ntica es opcional y tiene las limitaciones de la secciÃ³n 5.